# Solving a Real Physics PDE: The 1D Heat Equation

This notebook solves the one-dimensional heat equation

$$\frac{\partial u}{\partial t} = \kappa \frac{\partial^2 u}{\partial x^2}$$

on a finite interval with Dirichlet boundary conditions. After finite-difference discretization, the PDE solution is a matrix function:

$$u(t) = e^{-\kappa t L}u(0),$$

where `L` is the positive semidefinite discrete negative Laplacian. We approximate the heat semigroup with a bounded polynomial, which is the spectral transform QSVT would implement after a suitable block encoding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.polynomials import eval_polynomial
from qsvt.pde import dirichlet_laplacian_1d
from qsvt.spectral import (
    apply_function_to_hermitian,
    apply_polynomial_to_hermitian,
    eigh_hermitian,
)
from qsvt.templates import exponential_approximation_polynomial

np.set_printoptions(precision=4, suppress=True)

## Discretize the PDE

With zero boundary values at `x = 0` and `x = 1`, the interior grid satisfies

$$\frac{d u}{dt} = -\kappa L u,$$

where `L` is the standard second-difference negative Laplacian.

In [ ]:
n_points = 24
kappa = 0.015
time = 0.12

x, L = dirichlet_laplacian_1d(n_points)
eigenvalues, _ = eigh_hermitian(L)

eigenvalues[:5], eigenvalues[-1]

## Initial temperature profile

The initial condition combines a broad low-frequency mode with a narrow hot spot. Heat flow damps high-frequency components faster than low-frequency components.

In [ ]:
u0 = np.sin(np.pi * x) + 0.55 * np.exp(-180.0 * (x - 0.68) ** 2)
u0 = u0 / np.max(np.abs(u0))

plt.figure(figsize=(7, 3.5))
plt.plot(x, u0, marker="o")
plt.xlabel("position")
plt.ylabel("temperature")
plt.title("Initial heat profile")
plt.show()

## Rescale the Laplacian to `[-1, 1]`

The heat operator `exp(-kappa * time * L)` decays with increasing Laplacian eigenvalue. Map the smallest eigenvalue to `+1` and the largest to `-1`:

$$A = \frac{cI - L}{h}, \qquad c=\frac{\lambda_{\min}+\lambda_{\max}}{2}, \quad h=\frac{\lambda_{\max}-\lambda_{\min}}{2}.$$

Then

$$e^{-\kappa t L} = e^{-\kappa t c + \beta}\,p(A), \qquad \beta=\kappa t h,$$

where `p(x)` approximates `exp(beta*x - beta)`.

In [ ]:
lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]
center = 0.5 * (lambda_min + lambda_max)
half_width = 0.5 * (lambda_max - lambda_min)

A = (center * np.eye(n_points) - L) / half_width
beta = kappa * time * half_width
prefactor = np.exp(-kappa * time * center + beta)

np.linalg.eigvalsh(A)[[0, -1]], beta, prefactor

## Polynomial heat step

A higher degree improves the approximation of the exponential on the whole spectral interval. The output is compared against exact diagonalization of the discretized PDE operator.

In [ ]:
degree = 18
coeffs = exponential_approximation_polynomial(degree=degree, beta=beta)

u_poly = prefactor * (apply_polynomial_to_hermitian(A, coeffs) @ u0)
u_exact = apply_function_to_hermitian(L, lambda lam: np.exp(-kappa * time * lam)) @ u0

relative_error = np.linalg.norm(u_poly - u_exact) / np.linalg.norm(u_exact)
relative_error

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(x, u0, "o-", label="initial")
axes[0].plot(x, u_exact, "o-", label="exact heat solution")
axes[0].plot(x, u_poly, "--", label="polynomial solution")
axes[0].set_xlabel("position")
axes[0].set_ylabel("temperature")
axes[0].legend()

xs = np.linspace(-1, 1, 600)
target = np.exp(beta * xs - beta)
axes[1].plot(xs, target, label="bounded heat target")
axes[1].plot(xs, eval_polynomial(coeffs, xs), "--", label="polynomial")
axes[1].set_xlabel("rescaled Laplacian eigenvalue")
axes[1].set_ylabel("decay weight")
axes[1].legend()

fig.suptitle(
    f"1D heat equation via polynomial spectral transform, relative error={relative_error:.2e}"
)
plt.tight_layout()
plt.show()

## What this demonstrates

A linear physics PDE becomes a matrix-function problem after discretization. The current package can already express this workflow: construct a Hermitian operator, rescale its spectrum, approximate the target function with a bounded polynomial, and apply that polynomial spectrally for validation.